In [1]:
import Pkg; Pkg.add("ITensors")

    Updating registry at `C:\Users\AVM11\.julia\registries\General.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\AVM11\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\AVM11\.julia\environments\v1.12\Manifest.toml`


In [2]:
using DelimitedFiles
using Pkg
Pkg.activate("../")  # Activate the main project
using Dleto
using Random
using ITensors
using Statistics
using Plots
using Combinatorics

  Activating project at `c:\Users\AVM11\GitHub\OpenDleto`


WebIO._IJuliaInit()

Loading Dleto Plots Extension


In [3]:

function process_high_school_hypergraph(k::Int; seed::Union{Int,Nothing}=nothing, number_label::Union{Int,String}=1)
  
    # Load the contact-high-school temporal hypergraph dataset


    # Load the three data files
    nverts_file = "contact-high-school/contact-high-school-nverts.txt"
    simplices_file = "contact-high-school/contact-high-school-simplices.txt"
    times_file = "contact-high-school/contact-high-school-times.txt"

    # Read number of vertices per simplex
    nverts_full = vec(readdlm(nverts_file, Int))
    println("Total number of simplices: ", length(nverts_full))

    # Read all simplex node IDs (contiguous list)
    all_nodes_full = vec(readdlm(simplices_file, Int))
    println("Total node entries: ", length(all_nodes_full))

    # Read timestamps
    timestamps_full = vec(readdlm(times_file, Int))
    println("Total timestamps: ", length(timestamps_full))

    # Filter to random k nodes only
    all_unique_nodes = sort(unique(all_nodes_full))
    if seed !== nothing
        Random.seed!(seed)
        println("Random seed set to $seed")
    end
    random_k_nodes = Set(shuffle(all_unique_nodes)[1:k])
    println("Filtering to random $k nodes: $(sort(collect(random_k_nodes)))")

    # Filter simplices to only include those with all nodes in random_k_nodes
    filtered_indices = Int[]
    node_idx = 1
    for (simplex_idx, nvert) in enumerate(nverts_full)
        simplex_nodes = all_nodes_full[node_idx:(node_idx + nvert - 1)]
        if all(node in random_k_nodes for node in simplex_nodes)
            push!(filtered_indices, simplex_idx)
        end
        node_idx += nvert
    end

    # Create filtered datasets
    nverts = nverts_full[filtered_indices]
    timestamps = timestamps_full[filtered_indices]

    # Rebuild all_nodes for filtered simplices
    all_nodes = Int[]
    for simplex_idx in filtered_indices
        nvert = nverts_full[simplex_idx]
        orig_pos = sum(nverts_full[1:simplex_idx-1]) + 1
        simplex_nodes = all_nodes_full[orig_pos:(orig_pos + nvert - 1)]
        append!(all_nodes, simplex_nodes)
    end
    max_view = length(unique(all_nodes))
    println("\nFiltered dataset statistics:")
    println("Number of timestamped simplices: ", length(nverts))
    println("Number of unique node IDs: ", length(unique(all_nodes)))
    println("Simplex size distribution:")
    size_counts = Dict{Int, Int}()
    for size in nverts
        size_counts[size] = get(size_counts, size, 0) + 1
    end
    for size in sort(collect(keys(size_counts)))
        count = size_counts[size]
        println("  Size $size: $count simplices")
    end

    # Parse simplices from the contiguous node list
    simplices = Vector{Vector{Int}}()
    node_idx = 1
    for nvert in nverts
        simplex_nodes = all_nodes[node_idx:(node_idx + nvert - 1)]
        push!(simplices, simplex_nodes)
        node_idx += nvert
    end
    println("Parsed ", length(simplices), " simplices")
    println("First few simplices:")
    for i in 1:min(10, length(simplices))
        println("  Simplex $i (time $(timestamps[i])): $(simplices[i])")
    end

    # Create simple temporal simplex structure for processing
    temporal_simplices = [(simplices[i], timestamps[i], nverts[i]) for i in 1:length(simplices)]
    println("Created temporal hypergraph with ", length(temporal_simplices), " temporal simplices")

    # Basic statistics
    all_unique_nodes = sort(unique(all_nodes))
    println("Number of nodes: ", length(all_unique_nodes))
    println("Time range: $(minimum(timestamps)) to $(maximum(timestamps))")
    println("\nReady for tensor creation!")

    # Create a 3-mode tensor by decomposing hyperedges into 3-node subsets
    println("Creating 3-mode tensor from hyperedge decomposition ($k nodes only)...")
    unique_nodes = all_unique_nodes[1:max_view]
    node_to_idx = Dict(node => i for (i, node) in enumerate(unique_nodes))
    n_nodes = max_view
    println("Tensor dimensions:")
    println("  Node 1: $n_nodes")
    println("  Node 2: $n_nodes")
    println("  Node 3: $n_nodes")
    println("  Total entries: $(n_nodes^3)")
    println("  Using nodes: $(minimum(unique_nodes)) to $(maximum(unique_nodes))")

    function get_3node_combinations(nodes)
        if length(nodes) < 3
            return []
        elseif length(nodes) == 3
            return [nodes]
        else
            combinations = []
            for i in 1:length(nodes)-2
                for j in i+1:length(nodes)-1
                    for k in j+1:length(nodes)
                        push!(combinations, [nodes[i], nodes[j], nodes[k]])
                    end
                end
            end
            return combinations
        end
    end

    # Create ITensor indices for the 3-mode tensor
    i = Index(n_nodes, "node1")
    j = Index(n_nodes, "node2")
    k = Index(n_nodes, "node3")
    Triangles = ITensor(Float32, i, j, k)

    # Populate the tensor by counting 3-node combinations
    processed_simplices = 0
    total_triangles = 0
    skipped_small = 0
    for (idx, ts) in enumerate(temporal_simplices)
        triangle_combinations = get_3node_combinations(ts[1]) # ts[1] is the nodes vector
        if isempty(triangle_combinations)
            skipped_small += 1
            continue
        end
        for triangle in triangle_combinations
            node_indices = []
            for node in triangle
                if node in keys(node_to_idx) # Double check that all nodes are within our random sample
                    push!(node_indices, node_to_idx[node])
                end

            end
            if length(node_indices) == 3
                a, b, c = node_indices[1], node_indices[2], node_indices[3]
                Triangles[i=>a, j=>b, k=>c] += 1.0
                Triangles[i=>a, j=>c, k=>b] += 1.0
                Triangles[i=>b, j=>a, k=>c] += 1.0
                Triangles[i=>b, j=>c, k=>a] += 1.0
                Triangles[i=>c, j=>a, k=>b] += 1.0
                Triangles[i=>c, j=>b, k=>a] += 1.0
                total_triangles += 1
            end
        end
        processed_simplices += 1
    end

    
    # After populating Triangles and before tensor analysis
    # Output filtered hyperedges (unique, no permutations)
    hyperedges_filename = "filtered_hyperedges_$(string(number_label)).txt" 
    written = Set{Tuple{Int,Int,Int}}()   
    open(hyperedges_filename, "w") do io
        for a in 1:n_nodes, b in 1:n_nodes, c in 1:n_nodes
            if Triangles[i=>a, j=>b, k=>c] == 1.0
                edge = sort((a, b, c)) # sort to ignore permutations
                if !(edge in written)
                    print(io, "[", join(edge, ","), "]; ")
                    push!(written, edge)
                end
            end
        end
    end
    println("Filtered hyperedges saved to $hyperedges_filename")

end

process_high_school_hypergraph (generic function with 1 method)

In [4]:
process_high_school_hypergraph(327; number_label=327)

Total number of simplices: 172035
Total node entries: 352718
Total timestamps: 172035
Filtering to random 327 nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 